# Laya fine-tune: win / loss of backtest trades (Colab T4)

1. **Runtime -> Change runtime type -> T4 GPU.**
2. On the Mac: `bash scripts/pack_data.sh`, then upload `laya_data.tar.gz` to Google Drive at `MyDrive/laya/laya_data.tar.gz`.
3. Run the cells top to bottom. Checkpoints and reports are written to `MyDrive/laya/runs/<RUN_NAME>/`.

**Disconnected?** Reconnect, run all cells again with the same `RUN_NAME`: training resumes from the last save (every 20 min and every epoch end).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Get / update the code
!test -d /content/laya_finetune && git -C /content/laya_finetune pull --ff-only || git clone https://github.com/cavaldos/laya_finetune.git /content/laya_finetune
%cd /content/laya_finetune

In [ ]:
!bash scripts/colab_setup.sh

## Train

`RUN_NAME`: same name = resume. Extra `train.py` options go after the script, e.g.
`--anonymize`, `--epochs 2`, `--max-len 1024`, `--limit-train 2000` (quick try), `--loss soft-ce`.
`MAX_HOURS`: stop cleanly after N hours (free Colab sessions end after a few hours).

In [ ]:
%env RUN_NAME=run1
%env DRIVE=/content/drive/MyDrive/laya
%env MAX_HOURS=0
!bash scripts/colab_train.sh

## Result

In [ ]:
!cat /content/drive/MyDrive/laya/runs/$RUN_NAME/test_report.txt

In [ ]:
import json, os, pandas as pd
run = f"/content/drive/MyDrive/laya/runs/{os.environ['RUN_NAME']}"
progress = json.load(open(f"{run}/state/progress.json"))
display(pd.DataFrame(progress["history"]))        # loss / val per epoch
pred = pd.read_csv(f"{run}/predictions.csv")
display(pred.sort_values("p_win_laya_finetuned", ascending=False).head(20))

## Re-score a finished run (optional)

In [ ]:
!python train.py --eval-only /content/drive/MyDrive/laya/runs/$RUN_NAME